# 🧹 04 — Nettoyage final des données

Imputation des valeurs manquantes de `df_model.parquet` et sauvegarde de
`df_model_clean.parquet`, prêt pour la modélisation. Séparé de l'EDA
(`03_eda.ipynb`) : ici on prépare le dataset, on ne fait pas de visualisation.

In [ ]:
import os
import sys
from pathlib import Path
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, str(Path.cwd()))

import numpy as np
import pandas as pd
from src.config import RAW_DIR, TABLES_DIR, ANNEE_DEBUT, ANNEE_FIN, DEPTS

df = pd.read_parquet(TABLES_DIR / "df_model.parquet")
print(f"df_model chargé : {df.shape[0]:,} lignes × {df.shape[1]} colonnes")

---
## Imputation et sauvegarde du dataset nettoyé

In [2]:
df_clean = df.copy()

# ── Météo : médiane mensuelle nationale ──────────────────────────────────────
# Note : depuis le passage à Météo France MENSQ (voir 01_data_pipeline.ipynb),
# ces colonnes sont déjà à 0% de NaN (96/96 depts couverts). Ce fillna ne fait
# donc plus rien en pratique — conservé par sécurité si la source venait à
# changer à nouveau.
for col in ["temp_moy", "temp_min", "temp_max",
            "humidite_moy", "vent_moy", "precip_total"]:
    if col in df_clean.columns:
        mediane = df_clean.groupby("mois")[col].transform("median")
        df_clean[col] = df_clean[col].fillna(mediane)

# ── Qualité de l'air : médiane mensuelle nationale ────────────────────────────
for col in ["pm10_moy", "pm25_moy", "no2_moy", "o3_moy",
            "no_moy", "so2_moy", "nb_jours_pm10_eleve_est"]:
    if col in df_clean.columns:
        mediane = df_clean.groupby("mois")[col].transform("median")
        df_clean[col] = df_clean[col].fillna(mediane)

# ── Pollen + moisissures : médiane mensuelle + 0 hors saison ──────────────────
# (moisissure_alternaria_moy / moisissure_cladosporium_moy suivent la même
# logique que les pollens : mêmes fichiers RNSA, même limite de couverture)
for col in [c for c in df_clean.columns
            if c.startswith("pollen_") or c.startswith("moisissure_")]:
    mediane = df_clean.groupby("mois")[col].transform("median")
    df_clean[col] = df_clean[col].fillna(mediane).fillna(0)

# nb_jours_eleve n'a pas le préfixe "pollen_" et n'était donc jamais rempli
# (0 = pas de jour à pollen élevé détecté, cohérent avec l'absence de mesure)
if "nb_jours_eleve" in df_clean.columns:
    df_clean["nb_jours_eleve"] = df_clean["nb_jours_eleve"].fillna(0)

# ── Vérification finale ───────────────────────────────────────────────────────
missing_after = df_clean.isnull().sum()
missing_after = missing_after[missing_after > 0]

print("Valeurs manquantes restantes :")
if missing_after.empty:
    print("  ✅ Aucune — dataset complet !")
else:
    for col, n in missing_after.items():
        pct = n / len(df_clean) * 100
        print(f"  ⚠️  {col:<40} {n:>5} lignes ({pct:.1f}%)")

# Sauvegarde
df_clean.to_parquet(TABLES_DIR / "df_model_clean.parquet", index=False)
print(f"\n✅ Sauvegardé → data/processed/df_model_clean.parquet")
print(f"   {df_clean.shape[0]:,} lignes × {df_clean.shape[1]} colonnes")
print("\n🚀 Prêt pour la modélisation")

Valeurs manquantes restantes :
  ⚠️  taux_hosp_allergie                           5 lignes (0.1%)
  ⚠️  taux_sos_allergie                         3699 lignes (53.5%)
  ⚠️  taux_hosp_asthme                             5 lignes (0.1%)
  ⚠️  taux_sos_asthme                           3699 lignes (53.5%)
  ⚠️  taux_urgences_bronchiolite                   3 lignes (0.0%)
  ⚠️  taux_hosp_bronchiolite                     238 lignes (3.4%)
  ⚠️  taux_sos_bronchiolite                     3702 lignes (53.6%)

✅ Sauvegardé → data/processed/df_model_clean.parquet
   6,912 lignes × 61 colonnes

🚀 Prêt pour la modélisation
